<a href="https://colab.research.google.com/github/jansetaksoy/AD_Classification_Study/blob/scripts/AD_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# New Section

In [22]:
# ============================================================
# Alzheimer's Disease Classification Study
# End-to-end comparative machine-learning pipeline
#
# Study objective:
# To evaluate whether structured clinical, cognitive, and
# health-related variables can classify Alzheimer's disease
# diagnosis, and to determine whether adding broader
# risk-factor variables improves predictive performance
# relative to a focused clinical/cognitive feature set.
#
# Specific aims:
# Aim 1. Build and quality-control the analytic dataset.
# Aim 2. Train and compare supervised classification models
#         across predefined feature sets.
# Aim 3. Evaluate robustness, interpretability, and
#         limitations using threshold analysis, calibration,
#         subgroup summaries, and feature importance.
#
# Study framing:
# This script is written as a retrospective computational
# classification study and proof-of-concept analysis.
# It emphasizes transparent evaluation, comparison of
# predefined feature sets, and explicit discussion of
# robustness and limitations rather than claims of immediate
# clinical deployment.
# ============================================================

import os
import json
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import mannwhitneyu, chi2_contingency, spearmanr

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV,
    cross_val_score,
    learning_curve
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    f1_score,
    confusion_matrix,
    roc_curve,
    precision_score,
    recall_score,
    precision_recall_curve,
    average_precision_score,
    classification_report,
    brier_score_loss
)
from sklearn.calibration import calibration_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance
import joblib

# ------------------------------------------------------------
# 0. Study configuration
# ------------------------------------------------------------
INPUT_CSV = "alzheimers_disease_data.csv"
OUTPUT_DIR = "alzheimers_classification_study_outputs"

RANDOM_STATE = 42
TEST_SIZE = 0.20
VAL_SIZE_WITHIN_TEMP = 0.25   # yields 60/20/20 train/val/test
N_BOOTSTRAP = 500
N_CV_SPLITS = 5
MIN_SUBGROUP_N = 10
RUN_GRADIENT_BOOSTING = True

LABEL_COL = "Diagnosis"

# Variables excluded from modeling if present
EXCLUDE_ALWAYS = ["PatientID", "DoctorInCharge"]

# Potential low-cardinality or categorical variables encoded numerically
POTENTIAL_BINARY_OR_CATEGORICAL = [
    "Gender", "Ethnicity", "EducationLevel",
    "Smoking", "MemoryComplaints", "BehavioralProblems", "Confusion",
    "Disorientation", "PersonalityChanges", "DifficultyCompletingTasks",
    "Forgetfulness", "FamilyHistoryAlzheimers", "CardiovascularDisease",
    "Diabetes", "Depression", "HeadInjury", "Hypertension"
]

os.makedirs(OUTPUT_DIR, exist_ok=True)
sns.set_theme(style="whitegrid", context="talk")

# ------------------------------------------------------------
# 1. Aim 1: Load the analytic dataset
# ------------------------------------------------------------
print("\n=== Aim 1: Loading analytic dataset ===")
df = pd.read_csv(INPUT_CSV)

print("Raw dataset shape:", df.shape)
print("Available columns:")
print(df.columns.tolist())

with open(os.path.join(OUTPUT_DIR, "available_columns.json"), "w") as f:
    json.dump(df.columns.tolist(), f, indent=2)

# ------------------------------------------------------------
# 2. Define binary study outcome
# ------------------------------------------------------------
print("\n=== Defining binary study outcome ===")
df["label"] = pd.to_numeric(df[LABEL_COL], errors="coerce")
df = df.dropna(subset=["label"]).copy()
df["label"] = df["label"].astype(int)

if df["label"].nunique() < 2:
    raise ValueError("The study outcome contains only one class. Check the input data and outcome column.")

print("Outcome distribution:")
print(df["label"].value_counts())

outcome_distribution = pd.DataFrame({
    "class": df["label"].value_counts().index,
    "count": df["label"].value_counts().values,
    "percent": (100 * df["label"].value_counts(normalize=True).values).round(2)
})
outcome_distribution.to_csv(
    os.path.join(OUTPUT_DIR, "outcome_class_distribution.csv"),
    index=False
)

# ------------------------------------------------------------
# 3. Define predefined analytic feature sets
# ------------------------------------------------------------
print("\n=== Predefined analytic feature sets ===")

clinical_cognitive = [
    "Age", "Gender", "EducationLevel", "MMSE", "FunctionalAssessment",
    "MemoryComplaints", "BehavioralProblems", "ADL", "Confusion",
    "Disorientation", "PersonalityChanges", "DifficultyCompletingTasks",
    "Forgetfulness"
]

risk_factors = [
    "BMI", "Smoking", "AlcoholConsumption", "PhysicalActivity",
    "DietQuality", "SleepQuality", "FamilyHistoryAlzheimers",
    "CardiovascularDisease", "Diabetes", "Depression", "HeadInjury",
    "Hypertension", "SystolicBP", "DiastolicBP", "CholesterolTotal",
    "CholesterolLDL", "CholesterolHDL", "CholesterolTriglycerides"
]

clinical_cognitive = [c for c in clinical_cognitive if c in df.columns]
risk_factors = [c for c in risk_factors if c in df.columns]

feature_sets = {
    "core_clinical_cognitive": clinical_cognitive,
    "clinical_cognitive_plus_risk_factors": sorted(list(set(clinical_cognitive + risk_factors)))
}

for name, feats in feature_sets.items():
    print(f"{name}: {len(feats)} variables")

if len(clinical_cognitive) < 2:
    raise ValueError("Too few expected predictor variables were found. Check the dataset column names.")

with open(os.path.join(OUTPUT_DIR, "feature_set_definitions.json"), "w") as f:
    json.dump(feature_sets, f, indent=2)

# ------------------------------------------------------------
# 4. Construct analytic dataframe
# ------------------------------------------------------------
print("\n=== Constructing analytic dataframe ===")

all_features = sorted(list(set(clinical_cognitive + risk_factors)))
drop_candidates = [c for c in EXCLUDE_ALWAYS if c in df.columns]
print("Non-modeling columns identified for exclusion if present:", drop_candidates)

analytic_df = df[all_features + ["label"]].copy()

for c in all_features:
    analytic_df[c] = pd.to_numeric(analytic_df[c], errors="coerce")

# ------------------------------------------------------------
# 5. Aim 1: Dataset quality-control assessment
# ------------------------------------------------------------
print("\n=== Aim 1: Dataset quality-control assessment ===")

qc_rows = []
for col in analytic_df.columns:
    missing_n = analytic_df[col].isna().sum()
    qc_rows.append({
        "variable": col,
        "dtype": str(analytic_df[col].dtype),
        "missing_n": int(missing_n),
        "missing_percent": round(100 * missing_n / len(analytic_df), 2),
        "n_unique_nonmissing": int(analytic_df[col].dropna().nunique())
    })

qc_df = pd.DataFrame(qc_rows).sort_values("missing_percent", ascending=False)
qc_df.to_csv(os.path.join(OUTPUT_DIR, "analytic_dataset_qc.csv"), index=False)

# Duplicate-row screen
duplicate_n = int(analytic_df.duplicated().sum())
duplicate_pct = round(100 * duplicate_n / len(analytic_df), 2)
duplicate_report = pd.DataFrame([{
    "n_rows": len(analytic_df),
    "duplicate_rows": duplicate_n,
    "duplicate_percent": duplicate_pct
}])
duplicate_report.to_csv(os.path.join(OUTPUT_DIR, "duplicate_row_screen.csv"), index=False)

# Screen for possible leakage using correlation with outcome
leakage_rows = []
for c in all_features:
    temp = analytic_df[[c, "label"]].dropna()
    if temp[c].nunique() > 1:
        corr = temp[c].corr(temp["label"])
    else:
        corr = np.nan
    leakage_rows.append({
        "variable": c,
        "corr_with_outcome": corr,
        "abs_corr_with_outcome": np.abs(corr) if pd.notnull(corr) else np.nan
    })

leakage_df = pd.DataFrame(leakage_rows).sort_values("abs_corr_with_outcome", ascending=False)
leakage_df.to_csv(os.path.join(OUTPUT_DIR, "outcome_correlation_leakage_screen.csv"), index=False)

# Missingness figure
plt.figure(figsize=(10, max(4, 0.35 * len(qc_df))))
sns.barplot(
    data=qc_df[qc_df["variable"] != "label"],
    x="missing_percent", y="variable", color="#4472C4"
)
plt.xlabel("Missing percentage")
plt.ylabel("Variable")
plt.title("Missingness Across Analytic Variables")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure_missingness_across_variables.png"), dpi=300, bbox_inches="tight")
plt.close()

# Cardinality figure
tmp_unique = qc_df[qc_df["variable"] != "label"].copy().sort_values("n_unique_nonmissing", ascending=False)
plt.figure(figsize=(10, max(4, 0.35 * len(tmp_unique))))
sns.barplot(data=tmp_unique, x="n_unique_nonmissing", y="variable", color="#70AD47")
plt.xlabel("Unique non-missing values")
plt.ylabel("Variable")
plt.title("Variable Cardinality")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure_variable_cardinality.png"), dpi=300, bbox_inches="tight")
plt.close()

# Correlation matrix
numeric_for_corr = [c for c in all_features if analytic_df[c].dropna().nunique() > 1]
if len(numeric_for_corr) >= 2:
    corr = analytic_df[numeric_for_corr].corr()
    corr.to_csv(os.path.join(OUTPUT_DIR, "analytic_variable_correlations.csv"))
    plt.figure(figsize=(14, 12))
    sns.heatmap(corr, annot=False, cmap="coolwarm", center=0, cbar_kws={"shrink": 0.8})
    plt.title("Correlation Matrix of Analytic Variables")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "figure_variable_correlation_matrix.png"), dpi=300, bbox_inches="tight")
    plt.close()

# Outcome distribution figure
plt.figure(figsize=(6, 5))
sns.countplot(x="label", data=analytic_df, palette=["#5B9BD5", "#C00000"])
plt.title("Binary Outcome Distribution")
plt.xlabel("Alzheimer's disease diagnosis label")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure_outcome_distribution.png"), dpi=300, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# 6. Exploratory univariate association analysis
# ------------------------------------------------------------
print("\n=== Exploratory univariate association analysis ===")

univariate_rows = []

for c in all_features:
    sub = analytic_df[[c, "label"]].dropna()
    if sub.empty or sub[c].nunique() <= 1:
        continue

    is_binary_or_lowcard = (c in POTENTIAL_BINARY_OR_CATEGORICAL) or (sub[c].nunique() <= 5)

    row = {
        "variable": c,
        "n_nonmissing": len(sub),
        "n_unique": int(sub[c].nunique()),
        "group0_mean": sub.loc[sub["label"] == 0, c].mean(),
        "group1_mean": sub.loc[sub["label"] == 1, c].mean(),
        "group0_median": sub.loc[sub["label"] == 0, c].median(),
        "group1_median": sub.loc[sub["label"] == 1, c].median(),
        "spearman_r": np.nan,
        "spearman_p": np.nan,
        "test_used": None,
        "test_p": np.nan
    }

    try:
        rho, p_s = spearmanr(sub[c], sub["label"])
        row["spearman_r"] = rho
        row["spearman_p"] = p_s
    except:
        pass

    try:
        if is_binary_or_lowcard:
            table = pd.crosstab(sub[c], sub["label"])
            if table.shape[0] >= 2 and table.shape[1] == 2:
                chi2, p_val, _, _ = chi2_contingency(table)
                row["test_used"] = "chi2"
                row["test_p"] = p_val
        else:
            x0 = sub.loc[sub["label"] == 0, c]
            x1 = sub.loc[sub["label"] == 1, c]
            if len(x0) > 0 and len(x1) > 0:
                _, p_val = mannwhitneyu(x0, x1, alternative="two-sided")
                row["test_used"] = "mannwhitney"
                row["test_p"] = p_val
    except:
        pass

    univariate_rows.append(row)

univariate_df = pd.DataFrame(univariate_rows).sort_values("test_p", na_position="last")
univariate_df.to_csv(os.path.join(OUTPUT_DIR, "exploratory_univariate_associations.csv"), index=False)

if not univariate_df.empty:
    top_uni = univariate_df.dropna(subset=["spearman_r"]).copy()
    top_uni["abs_r"] = top_uni["spearman_r"].abs()
    top_uni = top_uni.sort_values("abs_r", ascending=False).head(15).sort_values("spearman_r")
    plt.figure(figsize=(9, 7))
    plt.barh(top_uni["variable"], top_uni["spearman_r"], color="#ED7D31")
    plt.axvline(0, color="black", linewidth=1)
    plt.xlabel("Spearman correlation with outcome")
    plt.title("Top Exploratory Univariate Associations")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "figure_top_univariate_associations.png"), dpi=300, bbox_inches="tight")
    plt.close()

# ------------------------------------------------------------
# 7. Create train/validation/test partitions
# ------------------------------------------------------------
print("\n=== Creating analytic partitions: train / validation / test ===")

X_all = analytic_df.drop(columns=["label"]).copy()
y_all = analytic_df["label"].copy()

X_temp, X_test, y_temp, y_test = train_test_split(
    X_all, y_all,
    test_size=TEST_SIZE,
    stratify=y_all,
    random_state=RANDOM_STATE
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp,
    test_size=VAL_SIZE_WITHIN_TEMP,
    stratify=y_temp,
    random_state=RANDOM_STATE
)

print(f"Training set size:   {len(X_train)}")
print(f"Validation set size: {len(X_val)}")
print(f"Test set size:       {len(X_test)}")

partition_summary = pd.DataFrame({
    "partition": ["train", "validation", "test"],
    "n": [len(X_train), len(X_val), len(X_test)],
    "positive_n": [int(y_train.sum()), int(y_val.sum()), int(y_test.sum())],
    "positive_percent": [
        round(100 * y_train.mean(), 2),
        round(100 * y_val.mean(), 2),
        round(100 * y_test.mean(), 2)
    ]
})
partition_summary.to_csv(os.path.join(OUTPUT_DIR, "analytic_partition_summary.csv"), index=False)

# ------------------------------------------------------------
# 8. Helper functions for evaluation
# ------------------------------------------------------------
def sensitivity_specificity(cm):
    if cm.shape != (2, 2):
        return np.nan, np.nan
    tn, fp, fn, tp = cm.ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    return sensitivity, specificity

def bootstrap_metric_ci(y_true, y_proba, threshold=0.5, n_bootstrap=500, random_state=42):
    rng = np.random.RandomState(random_state)
    aucs, accs, f1s, senss, specs, precs, recalls, briers, aps = [], [], [], [], [], [], [], [], []

    y_true = np.array(y_true)
    y_proba = np.array(y_proba)

    for _ in range(n_bootstrap):
        idx = rng.choice(np.arange(len(y_true)), size=len(y_true), replace=True)
        y_b = y_true[idx]
        p_b = y_proba[idx]

        if len(np.unique(y_b)) < 2:
            continue

        pred_b = (p_b >= threshold).astype(int)
        cm_b = confusion_matrix(y_b, pred_b)
        sens_b, spec_b = sensitivity_specificity(cm_b)

        aucs.append(roc_auc_score(y_b, p_b))
        accs.append(accuracy_score(y_b, pred_b))
        f1s.append(f1_score(y_b, pred_b, zero_division=0))
        senss.append(sens_b)
        specs.append(spec_b)
        precs.append(precision_score(y_b, pred_b, zero_division=0))
        recalls.append(recall_score(y_b, pred_b, zero_division=0))
        briers.append(brier_score_loss(y_b, p_b))
        aps.append(average_precision_score(y_b, p_b))

    def ci(arr):
        arr = np.array(arr)
        return np.nanpercentile(arr, 2.5), np.nanpercentile(arr, 97.5)

    return {
        "AUC_CI_low": ci(aucs)[0], "AUC_CI_high": ci(aucs)[1],
        "Accuracy_CI_low": ci(accs)[0], "Accuracy_CI_high": ci(accs)[1],
        "F1_CI_low": ci(f1s)[0], "F1_CI_high": ci(f1s)[1],
        "Sensitivity_CI_low": ci(senss)[0], "Sensitivity_CI_high": ci(senss)[1],
        "Specificity_CI_low": ci(specs)[0], "Specificity_CI_high": ci(specs)[1],
        "Precision_CI_low": ci(precs)[0], "Precision_CI_high": ci(precs)[1],
        "Recall_CI_low": ci(recalls)[0], "Recall_CI_high": ci(recalls)[1],
        "Brier_CI_low": ci(briers)[0], "Brier_CI_high": ci(briers)[1],
        "AP_CI_low": ci(aps)[0], "AP_CI_high": ci(aps)[1],
    }

def subgroup_metrics(y_true, y_proba, group_mask, group_name, threshold=0.5):
    y_true = np.array(y_true)
    y_proba = np.array(y_proba)
    mask = np.array(group_mask)

    n = int(mask.sum())
    if n < MIN_SUBGROUP_N:
        return {
            "Group": group_name,
            "N": n,
            "AUC": np.nan,
            "Accuracy": np.nan,
            "F1": np.nan,
            "Sensitivity": np.nan,
            "Specificity": np.nan,
            "AP": np.nan
        }

    y_g = y_true[mask]
    p_g = y_proba[mask]

    if len(np.unique(y_g)) < 2:
        return {
            "Group": group_name,
            "N": n,
            "AUC": np.nan,
            "Accuracy": np.nan,
            "F1": np.nan,
            "Sensitivity": np.nan,
            "Specificity": np.nan,
            "AP": np.nan
        }

    pred_g = (p_g >= threshold).astype(int)
    cm = confusion_matrix(y_g, pred_g)
    sens, spec = sensitivity_specificity(cm)

    return {
        "Group": group_name,
        "N": n,
        "AUC": roc_auc_score(y_g, p_g),
        "Accuracy": accuracy_score(y_g, pred_g),
        "F1": f1_score(y_g, pred_g, zero_division=0),
        "Sensitivity": sens,
        "Specificity": spec,
        "AP": average_precision_score(y_g, p_g)
    }

def build_preprocessor(feature_list):
    numeric_features = feature_list
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])
    return ColumnTransformer(
        transformers=[("num", numeric_transformer, numeric_features)],
        remainder="drop"
    )

def get_model_spaces():
    return {
        "logistic_regression": {
            "estimator": LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                random_state=RANDOM_STATE
            ),
            "param_grid": {
                "model__C": [0.01, 0.1, 1, 10],
                "model__penalty": ["l2"],
                "model__solver": ["lbfgs"]
            }
        },
        "random_forest": {
            "estimator": RandomForestClassifier(
                random_state=RANDOM_STATE,
                class_weight="balanced"
            ),
            "param_grid": {
                "model__n_estimators": [200, 500],
                "model__max_depth": [None, 5, 10],
                "model__min_samples_split": [2, 5, 10],
                "model__min_samples_leaf": [1, 2, 4]
            }
        }
    }

if RUN_GRADIENT_BOOSTING:
    def maybe_add_gradient_boosting(model_spaces):
        model_spaces["gradient_boosting"] = {
            "estimator": GradientBoostingClassifier(random_state=RANDOM_STATE),
            "param_grid": {
                "model__n_estimators": [100, 200],
                "model__learning_rate": [0.03, 0.05, 0.1],
                "model__max_depth": [2, 3]
            }
        }
        return model_spaces
else:
    def maybe_add_gradient_boosting(m):
        return m

def make_pipeline(preprocessor, estimator):
    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", estimator)
    ])

def find_best_threshold(y_true, y_proba, metric="f1"):
    thresholds = np.linspace(0.05, 0.95, 181)
    best_t = 0.5
    best_score = -np.inf

    for t in thresholds:
        pred = (y_proba >= t).astype(int)
        if metric == "f1":
            score = f1_score(y_true, pred, zero_division=0)
        elif metric == "youden":
            cm = confusion_matrix(y_true, pred)
            sens, spec = sensitivity_specificity(cm)
            score = sens + spec - 1
        else:
            raise ValueError("metric must be 'f1' or 'youden'")

        if score > best_score:
            best_score = score
            best_t = t

    return best_t, best_score

def net_benefit(y_true, y_proba, thresholds):
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    n = len(y_true)
    out = []
    for pt in thresholds:
        pred = (y_proba >= pt).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
        nb = (tp / n) - (fp / n) * (pt / (1 - pt))
        out.append(nb)
    return np.array(out)

# ------------------------------------------------------------
# 9. Baseline comparator analysis
# ------------------------------------------------------------
print("\n=== Baseline comparator analysis ===")

majority_class = y_train.mode()[0]
baseline_proba_val = np.full(shape=len(y_val), fill_value=float(y_train.mean()))
baseline_proba_test = np.full(shape=len(y_test), fill_value=float(y_train.mean()))

baseline_pred_test = np.full(shape=len(y_test), fill_value=int(majority_class))
cm_baseline = confusion_matrix(y_test, baseline_pred_test)
sens_b, spec_b = sensitivity_specificity(cm_baseline)

baseline_results = {
    "Model": "baseline_majority_class",
    "FeatureSet": "none",
    "BestParams": "{}",
    "CV_AUC_mean": np.nan,
    "CV_AUC_std": np.nan,
    "Validation_Selected_Threshold_F1": 0.5,
    "Validation_Selected_Threshold_Youden": 0.5,
    "Test_AUC": roc_auc_score(y_test, baseline_proba_test),
    "Test_AP": average_precision_score(y_test, baseline_proba_test),
    "Test_Accuracy": accuracy_score(y_test, baseline_pred_test),
    "Test_F1": f1_score(y_test, baseline_pred_test, zero_division=0),
    "Test_Precision": precision_score(y_test, baseline_pred_test, zero_division=0),
    "Test_Recall": recall_score(y_test, baseline_pred_test, zero_division=0),
    "Test_Sensitivity": sens_b,
    "Test_Specificity": spec_b,
    "Test_Brier": brier_score_loss(y_test, baseline_proba_test)
}
baseline_results.update(
    bootstrap_metric_ci(
        y_test.values, baseline_proba_test,
        threshold=0.5, n_bootstrap=N_BOOTSTRAP, random_state=RANDOM_STATE
    )
)

# ------------------------------------------------------------
# 10. Aim 2: Train and compare supervised classification models
# ------------------------------------------------------------
print("\n=== Aim 2: Model training and comparative evaluation ===")

cv = StratifiedKFold(n_splits=N_CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

all_results = [baseline_results]
subgroup_rows = []
best_estimators = {}
roc_curves = []
pr_curves = []
confusion_info = []
calibration_info = []
decision_curve_info = []
threshold_info = []

model_spaces = maybe_add_gradient_boosting(get_model_spaces())

for feature_set_name, feats in feature_sets.items():
    if len(feats) < 2:
        print(f"Skipping {feature_set_name}: too few predictor variables were available.")
        continue

    print(f"\n--- Evaluating feature configuration: {feature_set_name} ---")
    print(f"Number of predictors: {len(feats)}")

    X_train_fs = X_train[feats].copy()
    X_val_fs = X_val[feats].copy()
    X_test_fs = X_test[feats].copy()

    X_trainval_fs = pd.concat([X_train_fs, X_val_fs], axis=0)
    y_trainval_fs = pd.concat([y_train, y_val], axis=0)

    preprocessor = build_preprocessor(feats)

    for model_name, spec in model_spaces.items():
        print(f"\nTraining model family: {model_name}")

        pipe = make_pipeline(preprocessor, spec["estimator"])

        cv_scores = cross_val_score(
            pipe, X_train_fs, y_train,
            cv=cv, scoring="roc_auc", n_jobs=-1
        )

        grid = GridSearchCV(
            estimator=pipe,
            param_grid=spec["param_grid"],
            scoring="roc_auc",
            cv=cv,
            n_jobs=-1,
            refit=True
        )
        grid.fit(X_train_fs, y_train)

        best_train_model = grid.best_estimator_

        # Use validation data to select decision threshold
        val_proba = best_train_model.predict_proba(X_val_fs)[:, 1]

        best_t_f1, best_val_f1 = find_best_threshold(y_val.values, val_proba, metric="f1")
        best_t_youden, best_val_youden = find_best_threshold(y_val.values, val_proba, metric="youden")

        threshold_info.append({
            "FeatureSet": feature_set_name,
            "Model": model_name,
            "Validation_Selected_Threshold_F1": best_t_f1,
            "Validation_F1_at_Selected_Threshold": best_val_f1,
            "Validation_Selected_Threshold_Youden": best_t_youden,
            "Validation_Youden_at_Selected_Threshold": best_val_youden
        })

        # Refit final model on train + validation
        final_grid = GridSearchCV(
            estimator=pipe,
            param_grid=spec["param_grid"],
            scoring="roc_auc",
            cv=cv,
            n_jobs=-1,
            refit=True
        )
        final_grid.fit(X_trainval_fs, y_trainval_fs)

        best_model = final_grid.best_estimator_
        best_estimators[(feature_set_name, model_name)] = best_model

        test_proba = best_model.predict_proba(X_test_fs)[:, 1]
        test_pred = (test_proba >= best_t_f1).astype(int)

        cm = confusion_matrix(y_test, test_pred)
        sens, spec_ = sensitivity_specificity(cm)

        result_row = {
            "Model": model_name,
            "FeatureSet": feature_set_name,
            "BestParams": str(final_grid.best_params_),
            "CV_AUC_mean": cv_scores.mean(),
            "CV_AUC_std": cv_scores.std(),
            "Validation_Selected_Threshold_F1": best_t_f1,
            "Validation_Selected_Threshold_Youden": best_t_youden,
            "Test_AUC": roc_auc_score(y_test, test_proba),
            "Test_AP": average_precision_score(y_test, test_proba),
            "Test_Accuracy": accuracy_score(y_test, test_pred),
            "Test_F1": f1_score(y_test, test_pred, zero_division=0),
            "Test_Precision": precision_score(y_test, test_pred, zero_division=0),
            "Test_Recall": recall_score(y_test, test_pred, zero_division=0),
            "Test_Sensitivity": sens,
            "Test_Specificity": spec_,
            "Test_Brier": brier_score_loss(y_test, test_proba)
        }

        result_row.update(
            bootstrap_metric_ci(
                y_test.values, test_proba,
                threshold=best_t_f1,
                n_bootstrap=N_BOOTSTRAP,
                random_state=RANDOM_STATE
            )
        )
        all_results.append(result_row)

        # Save test-set predictions for traceability
        pred_df = pd.DataFrame({
            "y_true": y_test.values,
            "y_probability": test_proba,
            "y_predicted_at_validation_selected_threshold": test_pred
        })
        pred_df.to_csv(
            os.path.join(OUTPUT_DIR, f"test_set_predictions_{feature_set_name}_{model_name}.csv"),
            index=False
        )

        # ROC data
        fpr, tpr, _ = roc_curve(y_test, test_proba)
        roc_curves.append({
            "label": f"{feature_set_name} | {model_name} (AUC={result_row['Test_AUC']:.3f})",
            "fpr": fpr,
            "tpr": tpr
        })

        # PR data
        prec, rec, _ = precision_recall_curve(y_test, test_proba)
        pr_curves.append({
            "label": f"{feature_set_name} | {model_name} (AP={result_row['Test_AP']:.3f})",
            "precision": prec,
            "recall": rec
        })

        confusion_info.append({
            "title": f"{feature_set_name}\n{model_name}\nthreshold={best_t_f1:.2f}",
            "cm": cm
        })

        frac_pos, mean_pred = calibration_curve(y_test, test_proba, n_bins=10)
        calibration_info.append({
            "label": f"{feature_set_name} | {model_name}",
            "frac_pos": frac_pos,
            "mean_pred": mean_pred
        })

        thresholds_dca = np.linspace(0.05, 0.95, 50)
        decision_curve_info.append({
            "label": f"{feature_set_name} | {model_name}",
            "thresholds": thresholds_dca,
            "net_benefit": net_benefit(y_test.values, test_proba, thresholds_dca)
        })

        # Exploratory subgroup summaries
        if "Gender" in feats:
            for g_val in sorted(X_test_fs["Gender"].dropna().unique()):
                mask = X_test_fs["Gender"] == g_val
                subgroup_rows.append(
                    subgroup_metrics(
                        y_test.values,
                        test_proba,
                        mask,
                        f"{feature_set_name}|{model_name}|Gender={g_val}",
                        threshold=best_t_f1
                    )
                )

        if "Age" in feats:
            median_age = X_test_fs["Age"].median()
            mask_younger = X_test_fs["Age"] < median_age
            mask_older = X_test_fs["Age"] >= median_age
            subgroup_rows.append(
                subgroup_metrics(
                    y_test.values,
                    test_proba,
                    mask_younger,
                    f"{feature_set_name}|{model_name}|Age<{median_age:.1f}",
                    threshold=best_t_f1
                )
            )
            subgroup_rows.append(
                subgroup_metrics(
                    y_test.values,
                    test_proba,
                    mask_older,
                    f"{feature_set_name}|{model_name}|Age>={median_age:.1f}",
                    threshold=best_t_f1
                )
            )

        report = classification_report(y_test, test_pred, output_dict=True, zero_division=0)
        pd.DataFrame(report).transpose().to_csv(
            os.path.join(OUTPUT_DIR, f"classification_report_{feature_set_name}_{model_name}.csv")
        )

# ------------------------------------------------------------
# 11. Save comparative model results
# ------------------------------------------------------------
results_df = pd.DataFrame(all_results).sort_values("Test_AUC", ascending=False)
results_df.to_csv(os.path.join(OUTPUT_DIR, "comparative_model_results.csv"), index=False)

threshold_df = pd.DataFrame(threshold_info)
threshold_df.to_csv(os.path.join(OUTPUT_DIR, "validation_threshold_selection.csv"), index=False)

subgroup_df = pd.DataFrame(subgroup_rows)
if len(subgroup_df) > 0:
    subgroup_df.to_csv(os.path.join(OUTPUT_DIR, "exploratory_subgroup_performance_summary.csv"), index=False)

print("\n=== Comparative model performance summary ===")
print(results_df[[
    "Model", "FeatureSet", "CV_AUC_mean", "CV_AUC_std",
    "Test_AUC", "Test_AP", "Test_Accuracy", "Test_F1",
    "Test_Sensitivity", "Test_Specificity"
]].round(3))

# ------------------------------------------------------------
# 12. Figure: ROC curves
# ------------------------------------------------------------
plt.figure(figsize=(8, 8))
plt.plot([0, 1], [0, 1], "k--", label="No-discrimination line")
plt.plot(*roc_curve(y_test, baseline_proba_test)[:2],
         label=f"Baseline (AUC={baseline_results['Test_AUC']:.3f})")
for item in roc_curves:
    plt.plot(item["fpr"], item["tpr"], label=item["label"])
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC Curves Across Modeling Approaches")
plt.legend(loc="lower right", fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure_roc_curves.png"), dpi=300, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# 13. Figure: Precision-recall curves
# ------------------------------------------------------------
plt.figure(figsize=(8, 8))
baseline_precision = y_test.mean()
plt.axhline(baseline_precision, linestyle="--", color="black", label=f"Outcome prevalence={baseline_precision:.3f}")
for item in pr_curves:
    plt.plot(item["recall"], item["precision"], label=item["label"])
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves Across Modeling Approaches")
plt.legend(loc="lower left", fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure_precision_recall_curves.png"), dpi=300, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# 14. Figure: Confusion matrices
# ------------------------------------------------------------
n_plots = len(confusion_info) + 1
ncols = 2
nrows = int(np.ceil(n_plots / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(12, 4.5 * nrows))
axes = np.array(axes).reshape(-1)

cm0 = confusion_matrix(y_test, baseline_pred_test)
sns.heatmap(cm0, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0])
axes[0].set_title("baseline_majority_class")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Observed")

for i, item in enumerate(confusion_info, start=1):
    sns.heatmap(item["cm"], annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[i])
    axes[i].set_title(item["title"])
    axes[i].set_xlabel("Predicted")
    axes[i].set_ylabel("Observed")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure_confusion_matrices.png"), dpi=300, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# 15. Figure: Calibration curves
# ------------------------------------------------------------
plt.figure(figsize=(8, 8))
plt.plot([0, 1], [0, 1], "k--", label="Perfect calibration")
for item in calibration_info:
    plt.plot(item["mean_pred"], item["frac_pos"], marker="o", label=item["label"])
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed fraction positive")
plt.title("Calibration Curves")
plt.legend(fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure_calibration_curves.png"), dpi=300, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# 16. Figure: Decision curve analysis
# ------------------------------------------------------------
thresholds_dca = np.linspace(0.05, 0.95, 50)
prevalence = y_test.mean()
treat_all_nb = prevalence - (1 - prevalence) * (thresholds_dca / (1 - thresholds_dca))
treat_none_nb = np.zeros_like(thresholds_dca)

plt.figure(figsize=(9, 7))
plt.plot(thresholds_dca, treat_all_nb, linestyle="--", color="gray", label="Treat all")
plt.plot(thresholds_dca, treat_none_nb, linestyle="--", color="black", label="Treat none")
for item in decision_curve_info:
    plt.plot(item["thresholds"], item["net_benefit"], label=item["label"])
plt.xlabel("Threshold probability")
plt.ylabel("Net benefit")
plt.title("Decision Curve Analysis")
plt.legend(fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure_decision_curve_analysis.png"), dpi=300, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# 17. Figure: Learning curve for best non-baseline model
# ------------------------------------------------------------
best_nonbaseline = results_df[results_df["Model"] != "baseline_majority_class"].iloc[0]
best_fs = best_nonbaseline["FeatureSet"]
best_model_name = best_nonbaseline["Model"]
best_model = best_estimators[(best_fs, best_model_name)]
best_feats = feature_sets[best_fs]

train_sizes, train_scores, val_scores = learning_curve(
    best_model,
    X_temp[best_feats],
    y_temp,
    train_sizes=np.linspace(0.2, 1.0, 8),
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1
)

plt.figure(figsize=(8, 6))
plt.plot(train_sizes, train_scores.mean(axis=1), marker="o", label="Training AUC")
plt.plot(train_sizes, val_scores.mean(axis=1), marker="o", label="Validation AUC")
plt.fill_between(
    train_sizes,
    train_scores.mean(axis=1) - train_scores.std(axis=1),
    train_scores.mean(axis=1) + train_scores.std(axis=1),
    alpha=0.15
)
plt.fill_between(
    train_sizes,
    val_scores.mean(axis=1) - val_scores.std(axis=1),
    val_scores.mean(axis=1) + val_scores.std(axis=1),
    alpha=0.15
)
plt.xlabel("Training set size")
plt.ylabel("AUC")
plt.title(f"Learning Curve: {best_fs} | {best_model_name}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure_learning_curve_best_model.png"), dpi=300, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# 18. Aim 3: Model interpretation
# ------------------------------------------------------------
for (fs_name, model_name), est in best_estimators.items():
    feats = feature_sets[fs_name]
    transformed_feature_names = feats

    if model_name == "logistic_regression":
        coefs = est.named_steps["model"].coef_[0]
        coef_df = pd.DataFrame({
            "variable": transformed_feature_names,
            "coefficient": coefs,
            "abs_coefficient": np.abs(coefs)
        }).sort_values("abs_coefficient", ascending=False)
        coef_df.to_csv(os.path.join(OUTPUT_DIR, f"model_interpretation_logistic_coefficients_{fs_name}.csv"), index=False)

        top_df = coef_df.head(15).sort_values("coefficient")
        plt.figure(figsize=(8, 6))
        plt.barh(top_df["variable"], top_df["coefficient"], color="#5B9BD5")
        plt.axvline(0, color="black", linewidth=1)
        plt.xlabel("Coefficient")
        plt.title(f"Top Logistic Regression Coefficients: {fs_name}")
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, f"figure_logistic_coefficients_{fs_name}.png"), dpi=300, bbox_inches="tight")
        plt.close()

    elif model_name == "random_forest":
        imps = est.named_steps["model"].feature_importances_
        imp_df = pd.DataFrame({
            "variable": transformed_feature_names,
            "importance": imps
        }).sort_values("importance", ascending=False)
        imp_df.to_csv(os.path.join(OUTPUT_DIR, f"model_interpretation_random_forest_importance_{fs_name}.csv"), index=False)

        top_df = imp_df.head(15).sort_values("importance")
        plt.figure(figsize=(8, 6))
        plt.barh(top_df["variable"], top_df["importance"], color="#70AD47")
        plt.xlabel("Importance")
        plt.title(f"Top Random Forest Importances: {fs_name}")
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, f"figure_random_forest_importance_{fs_name}.png"), dpi=300, bbox_inches="tight")
        plt.close()

    elif model_name == "gradient_boosting":
        if hasattr(est.named_steps["model"], "feature_importances_"):
            imps = est.named_steps["model"].feature_importances_
            imp_df = pd.DataFrame({
                "variable": transformed_feature_names,
                "importance": imps
            }).sort_values("importance", ascending=False)
            imp_df.to_csv(os.path.join(OUTPUT_DIR, f"model_interpretation_gradient_boosting_importance_{fs_name}.csv"), index=False)

            top_df = imp_df.head(15).sort_values("importance")
            plt.figure(figsize=(8, 6))
            plt.barh(top_df["variable"], top_df["importance"], color="#FFC000")
            plt.xlabel("Importance")
            plt.title(f"Top Gradient Boosting Importances: {fs_name}")
            plt.tight_layout()
            plt.savefig(os.path.join(OUTPUT_DIR, f"figure_gradient_boosting_importance_{fs_name}.png"), dpi=300, bbox_inches="tight")
            plt.close()

perm = permutation_importance(
    best_model,
    X_test[best_feats],
    y_test,
    n_repeats=20,
    random_state=RANDOM_STATE,
    scoring="roc_auc"
)
perm_df = pd.DataFrame({
    "variable": best_feats,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)
perm_df.to_csv(os.path.join(OUTPUT_DIR, "model_interpretation_best_model_permutation_importance.csv"), index=False)

plt.figure(figsize=(8, 6))
top_perm = perm_df.head(15).sort_values("importance_mean")
plt.barh(top_perm["variable"], top_perm["importance_mean"], xerr=top_perm["importance_std"], color="#A5A5A5")
plt.xlabel("Permutation importance (AUC decrease)")
plt.title(f"Permutation Importance: Best Model\n{best_fs} | {best_model_name}")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure_best_model_permutation_importance.png"), dpi=300, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# 19. Figures: Distributions for selected variables
# ------------------------------------------------------------
top_variables_for_distribution = []
if not univariate_df.empty:
    top_variables_for_distribution = (
        univariate_df.dropna(subset=["spearman_r"])
        .assign(abs_r=lambda d: d["spearman_r"].abs())
        .sort_values("abs_r", ascending=False)["variable"]
        .head(6).tolist()
    )

if len(top_variables_for_distribution) < 6:
    top_variables_for_distribution = perm_df["variable"].head(6).tolist()

for feat in top_variables_for_distribution:
    plot_df = analytic_df[[feat, "label"]].dropna().copy()
    if plot_df.empty:
        continue

    if plot_df[feat].nunique() <= 8:
        plt.figure(figsize=(7, 5))
        prop_df = (
            plot_df.groupby([feat, "label"]).size()
            .reset_index(name="count")
        )
        sns.barplot(data=prop_df, x=feat, y="count", hue="label")
        plt.title(f"{feat} by Outcome Class")
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, f"figure_distribution_{feat}.png"), dpi=300, bbox_inches="tight")
        plt.close()
    else:
        plt.figure(figsize=(7, 5))
        sns.boxplot(data=plot_df, x="label", y=feat)
        sns.stripplot(
            data=plot_df.sample(min(500, len(plot_df)), random_state=RANDOM_STATE),
            x="label", y=feat, color="black", alpha=0.25, size=3
        )
        plt.title(f"{feat} by Outcome Class")
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, f"figure_distribution_{feat}.png"), dpi=300, bbox_inches="tight")
        plt.close()

# ------------------------------------------------------------
# 20. Exploratory subgroup performance summaries
# ------------------------------------------------------------
if len(subgroup_df) > 0:
    subgroup_plot_df = subgroup_df.dropna(subset=["AUC"]).copy()
    if not subgroup_plot_df.empty:
        top_show = subgroup_plot_df.head(20)
        plt.figure(figsize=(12, max(6, 0.4 * len(top_show))))
        sns.barplot(data=top_show, x="AUC", y="Group", color="#4472C4")
        plt.title("Exploratory Subgroup AUC Summaries")
        plt.xlim(0, 1)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "figure_subgroup_auc_summary.png"), dpi=300, bbox_inches="tight")
        plt.close()

# ------------------------------------------------------------
# 21. Figure: Threshold-performance profile for best model
# ------------------------------------------------------------
best_threshold = float(best_nonbaseline["Validation_Selected_Threshold_F1"])
best_test_preds = pd.read_csv(
    os.path.join(OUTPUT_DIR, f"test_set_predictions_{best_fs}_{best_model_name}.csv")
)
y_true_bt = best_test_preds["y_true"].values
y_proba_bt = best_test_preds["y_probability"].values

threshold_grid = np.linspace(0.05, 0.95, 181)
f1_list, sens_list, spec_list, acc_list = [], [], [], []

for t in threshold_grid:
    pred = (y_proba_bt >= t).astype(int)
    cm = confusion_matrix(y_true_bt, pred)
    sens, spec = sensitivity_specificity(cm)
    f1_list.append(f1_score(y_true_bt, pred, zero_division=0))
    sens_list.append(sens)
    spec_list.append(spec)
    acc_list.append(accuracy_score(y_true_bt, pred))

plt.figure(figsize=(9, 6))
plt.plot(threshold_grid, f1_list, label="F1")
plt.plot(threshold_grid, sens_list, label="Sensitivity")
plt.plot(threshold_grid, spec_list, label="Specificity")
plt.plot(threshold_grid, acc_list, label="Accuracy")
plt.axvline(best_threshold, linestyle="--", color="black", label=f"Validation-selected threshold={best_threshold:.2f}")
plt.xlabel("Classification threshold")
plt.ylabel("Metric value")
plt.title(f"Threshold Performance Profile: {best_fs} | {best_model_name}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure_best_model_threshold_profile.png"), dpi=300, bbox_inches="tight")
plt.close()

# ------------------------------------------------------------
# 22. Save best model artifact
# ------------------------------------------------------------
joblib.dump(best_model, os.path.join(OUTPUT_DIR, "best_model_artifact.joblib"))

best_model_summary = {
    "best_feature_set": best_fs,
    "best_model_name": best_model_name,
    "best_test_auc": float(best_nonbaseline["Test_AUC"]),
    "best_test_average_precision": float(best_nonbaseline["Test_AP"]),
    "validation_selected_threshold_f1": float(best_nonbaseline["Validation_Selected_Threshold_F1"])
}
with open(os.path.join(OUTPUT_DIR, "best_model_summary.json"), "w") as f:
    json.dump(best_model_summary, f, indent=2)

# ------------------------------------------------------------
# 25. Final console summary
# ------------------------------------------------------------
print("\n=== Study pipeline completed successfully ===")
print(f"Output directory: {OUTPUT_DIR}")

print("\nTop-performing models:")
print(results_df[[
    "Model", "FeatureSet", "Test_AUC", "Test_AP",
    "Test_Accuracy", "Test_F1", "Test_Sensitivity", "Test_Specificity"
]].head(10).round(3))

print("\nBest non-baseline model summary:")
print(best_model_summary)
# ------------------------------------------------------------
# 12. Generate and save visual figures for reporting
# ------------------------------------------------------------
print("\n=== Aim 3: Generating visual figures for reporting ===")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# -----------------------------
# Figure 1: Model comparison bar chart
# -----------------------------
plot_results = results_df[results_df["Model"] != "baseline_majority_class"].copy()

plt.figure(figsize=(12, 7))
plot_df = plot_results.melt(
    id_vars=["Model", "FeatureSet"],
    value_vars=["Test_AUC", "Test_AP", "Test_Accuracy", "Test_F1"],
    var_name="Metric",
    value_name="Score"
)
plot_df["ModelLabel"] = plot_df["Model"] + "\n" + plot_df["FeatureSet"]

sns.barplot(data=plot_df, x="Metric", y="Score", hue="ModelLabel")
plt.ylim(0, 1.05)
plt.title("Comparison of Test Performance Metrics Across Models")
plt.ylabel("Score")
plt.xlabel("Metric")
plt.legend(title="Model / Feature set", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure_model_comparison_barplot.png"), dpi=300, bbox_inches="tight")
plt.close()

# -----------------------------
# Figure 2: Test AUC only bar chart
# -----------------------------
auc_plot_df = plot_results.copy()
auc_plot_df["ModelLabel"] = auc_plot_df["Model"] + "\n" + auc_plot_df["FeatureSet"]

plt.figure(figsize=(10, 6))
sns.barplot(data=auc_plot_df, x="ModelLabel", y="Test_AUC", palette="viridis")
plt.ylim(0, 1.05)
plt.title("Test AUC by Model and Feature Set")
plt.xlabel("Model / Feature set")
plt.ylabel("Test AUC")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure_test_auc_barplot.png"), dpi=300, bbox_inches="tight")
plt.close()

# -----------------------------
# Figure 3: ROC curves
# -----------------------------
plt.figure(figsize=(8, 8))
plt.plot([0, 1], [0, 1], "k--", label="No-discrimination line")

for _, row in results_df.iterrows():
    model_name = row["Model"]
    feature_set = row["FeatureSet"]

    if model_name == "baseline_majority_class":
        baseline_proba = np.full(len(y_test), y_train.mean())
        fpr, tpr, _ = roc_curve(y_test, baseline_proba)
        auc_val = roc_auc_score(y_test, baseline_proba)
        plt.plot(fpr, tpr, label=f"Baseline (AUC={auc_val:.3f})")
    else:
        pred_path = os.path.join(OUTPUT_DIR, f"test_set_predictions_{feature_set}_{model_name}.csv")
        if os.path.exists(pred_path):
            pred_df = pd.read_csv(pred_path)
            fpr, tpr, _ = roc_curve(pred_df["y_true"], pred_df["y_probability"])
            auc_val = roc_auc_score(pred_df["y_true"], pred_df["y_probability"])
            plt.plot(fpr, tpr, label=f"{model_name} | {feature_set} (AUC={auc_val:.3f})")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves Across Models")
plt.legend(loc="lower right", fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure_roc_curves.png"), dpi=300, bbox_inches="tight")
plt.close()

# -----------------------------
# Figure 4: Precision-Recall curves
# -----------------------------
plt.figure(figsize=(8, 8))
baseline_precision = y_test.mean()
plt.axhline(baseline_precision, linestyle="--", color="black", label=f"Outcome prevalence={baseline_precision:.3f}")

for _, row in results_df.iterrows():
    model_name = row["Model"]
    feature_set = row["FeatureSet"]

    if model_name == "baseline_majority_class":
        continue

    pred_path = os.path.join(OUTPUT_DIR, f"test_set_predictions_{feature_set}_{model_name}.csv")
    if os.path.exists(pred_path):
        pred_df = pd.read_csv(pred_path)
        precision, recall, _ = precision_recall_curve(pred_df["y_true"], pred_df["y_probability"])
        ap = average_precision_score(pred_df["y_true"], pred_df["y_probability"])
        plt.plot(recall, precision, label=f"{model_name} | {feature_set} (AP={ap:.3f})")

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves Across Models")
plt.legend(loc="lower left", fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure_precision_recall_curves.png"), dpi=300, bbox_inches="tight")
plt.close()

# -----------------------------
# Figure 5: Confusion matrices
# -----------------------------
confusion_items = []

# Baseline
baseline_pred = np.full(shape=len(y_test), fill_value=int(y_train.mode()[0]))
baseline_cm = confusion_matrix(y_test, baseline_pred)
confusion_items.append(("baseline_majority_class", baseline_cm))

# Other models
for _, row in results_df.iterrows():
    model_name = row["Model"]
    feature_set = row["FeatureSet"]

    if model_name == "baseline_majority_class":
        continue

    pred_path = os.path.join(OUTPUT_DIR, f"test_set_predictions_{feature_set}_{model_name}.csv")
    if os.path.exists(pred_path):
        pred_df = pd.read_csv(pred_path)

        # Use threshold saved in results table
        thr = row["Validation_Selected_Threshold_F1"]
        y_pred = (pred_df["y_probability"] >= thr).astype(int)
        cm = confusion_matrix(pred_df["y_true"], y_pred)
        confusion_items.append((f"{model_name}\n{feature_set}", cm))

n_plots = len(confusion_items)
ncols = 2
nrows = int(np.ceil(n_plots / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(12, 5 * nrows))
axes = np.array(axes).reshape(-1)

for i, (title, cm) in enumerate(confusion_items):
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[i])
    axes[i].set_title(title)
    axes[i].set_xlabel("Predicted")
    axes[i].set_ylabel("Observed")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure_confusion_matrices.png"), dpi=300, bbox_inches="tight")
plt.close()

# -----------------------------
# Figure 6: Calibration curves
# -----------------------------
plt.figure(figsize=(8, 8))
plt.plot([0, 1], [0, 1], "k--", label="Perfect calibration")

for _, row in results_df.iterrows():
    model_name = row["Model"]
    feature_set = row["FeatureSet"]

    if model_name == "baseline_majority_class":
        continue

    pred_path = os.path.join(OUTPUT_DIR, f"test_set_predictions_{feature_set}_{model_name}.csv")
    if os.path.exists(pred_path):
        pred_df = pd.read_csv(pred_path)
        frac_pos, mean_pred = calibration_curve(pred_df["y_true"], pred_df["y_probability"], n_bins=10)
        plt.plot(mean_pred, frac_pos, marker="o", label=f"{model_name} | {feature_set}")

plt.xlabel("Mean Predicted Probability")
plt.ylabel("Observed Fraction Positive")
plt.title("Calibration Curves")
plt.legend(fontsize=8)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure_calibration_curves.png"), dpi=300, bbox_inches="tight")
plt.close()

# -----------------------------
# Figure 7: Best-model threshold profile
# -----------------------------
best_nonbaseline = results_df[results_df["Model"] != "baseline_majority_class"].iloc[0]
best_model_name = best_nonbaseline["Model"]
best_feature_set = best_nonbaseline["FeatureSet"]
best_threshold = float(best_nonbaseline["Validation_Selected_Threshold_F1"])

best_pred_path = os.path.join(
    OUTPUT_DIR,
    f"test_set_predictions_{best_feature_set}_{best_model_name}.csv"
)

if os.path.exists(best_pred_path):
    best_pred_df = pd.read_csv(best_pred_path)
    y_true_best = best_pred_df["y_true"].values
    y_proba_best = best_pred_df["y_probability"].values

    threshold_grid = np.linspace(0.05, 0.95, 181)
    f1_list, sens_list, spec_list, acc_list = [], [], [], []

    for t in threshold_grid:
        y_pred_t = (y_proba_best >= t).astype(int)
        cm_t = confusion_matrix(y_true_best, y_pred_t)
        sens_t, spec_t = sensitivity_specificity(cm_t)

        f1_list.append(f1_score(y_true_best, y_pred_t, zero_division=0))
        sens_list.append(sens_t)
        spec_list.append(spec_t)
        acc_list.append(accuracy_score(y_true_best, y_pred_t))

    plt.figure(figsize=(9, 6))
    plt.plot(threshold_grid, f1_list, label="F1")
    plt.plot(threshold_grid, sens_list, label="Sensitivity")
    plt.plot(threshold_grid, spec_list, label="Specificity")
    plt.plot(threshold_grid, acc_list, label="Accuracy")
    plt.axvline(best_threshold, linestyle="--", color="black", label=f"Selected threshold={best_threshold:.2f}")
    plt.xlabel("Classification threshold")
    plt.ylabel("Metric value")
    plt.title(f"Threshold Performance Profile\n{best_model_name} | {best_feature_set}")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, "figure_best_model_threshold_profile.png"), dpi=300, bbox_inches="tight")
    plt.close()

# -----------------------------
# Figure 8: Learning curve for best model
# -----------------------------
best_feats = feature_sets[best_feature_set]
best_model = best_estimators[(best_feature_set, best_model_name)]

train_sizes, train_scores, val_scores = learning_curve(
    best_model,
    X_temp[best_feats],
    y_temp,
    train_sizes=np.linspace(0.2, 1.0, 8),
    cv=StratifiedKFold(n_splits=N_CV_SPLITS, shuffle=True, random_state=RANDOM_STATE),
    scoring="roc_auc",
    n_jobs=-1
)

plt.figure(figsize=(8, 6))
plt.plot(train_sizes, train_scores.mean(axis=1), marker="o", label="Training AUC")
plt.plot(train_sizes, val_scores.mean(axis=1), marker="o", label="Validation AUC")
plt.fill_between(
    train_sizes,
    train_scores.mean(axis=1) - train_scores.std(axis=1),
    train_scores.mean(axis=1) + train_scores.std(axis=1),
    alpha=0.15
)
plt.fill_between(
    train_sizes,
    val_scores.mean(axis=1) - val_scores.std(axis=1),
    val_scores.mean(axis=1) + val_scores.std(axis=1),
    alpha=0.15
)
plt.xlabel("Training Set Size")
plt.ylabel("AUC")
plt.title(f"Learning Curve for Best Non-Baseline Model\n{best_model_name} | {best_feature_set}")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "figure_learning_curve_best_model.png"), dpi=300, bbox_inches="tight")
plt.close()

# -----------------------------
# Figure 9: Subgroup performance bar plot
# -----------------------------
subgroup_path = os.path.join(OUTPUT_DIR, "exploratory_subgroup_performance_summary.csv")
if os.path.exists(subgroup_path):
    subgroup_df = pd.read_csv(subgroup_path)
    subgroup_plot_df = subgroup_df.dropna(subset=["AUC"]).copy()

    if not subgroup_plot_df.empty:
        subgroup_plot_df = subgroup_plot_df.sort_values("AUC", ascending=False).head(20)

        plt.figure(figsize=(12, max(6, 0.4 * len(subgroup_plot_df))))
        sns.barplot(data=subgroup_plot_df, x="AUC", y="Group", color="#4472C4")
        plt.title("Exploratory Subgroup AUC Summary")
        plt.xlim(0, 1)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, "figure_subgroup_auc_summary.png"), dpi=300, bbox_inches="tight")
        plt.close()

print("Visual figures saved successfully in:", OUTPUT_DIR)


=== Aim 1: Loading analytic dataset ===
Raw dataset shape: (2149, 35)
Available columns:
['PatientID', 'Age', 'Gender', 'Ethnicity', 'EducationLevel', 'BMI', 'Smoking', 'AlcoholConsumption', 'PhysicalActivity', 'DietQuality', 'SleepQuality', 'FamilyHistoryAlzheimers', 'CardiovascularDisease', 'Diabetes', 'Depression', 'HeadInjury', 'Hypertension', 'SystolicBP', 'DiastolicBP', 'CholesterolTotal', 'CholesterolLDL', 'CholesterolHDL', 'CholesterolTriglycerides', 'MMSE', 'FunctionalAssessment', 'MemoryComplaints', 'BehavioralProblems', 'ADL', 'Confusion', 'Disorientation', 'PersonalityChanges', 'DifficultyCompletingTasks', 'Forgetfulness', 'Diagnosis', 'DoctorInCharge']

=== Defining binary study outcome ===
Outcome distribution:
label
0    1389
1     760
Name: count, dtype: int64

=== Predefined analytic feature sets ===
core_clinical_cognitive: 13 variables
clinical_cognitive_plus_risk_factors: 31 variables

=== Constructing analytic dataframe ===
Non-modeling columns identified for excl